In [ ]:
# 1. Mount Google Drive to Colab
from google.colab import drive
drive.mount('/content/drive')

# 2. Install the required database and tensor libraries in the background
!pip install lmdb einops

# 3. Clone the Restormer repository and navigate into the folder
!git clone https://github.com/swz30/Restormer.git
%cd Restormer

In [ ]:
import os

# 1. Prepare the target directory within Restormer
os.makedirs('demo_data', exist_ok=True)
!rm -rf demo_data/HIDE

# 2. Specify the path to the folder in Google Drive (update this if you placed it elsewhere in your Drive)
drive_path = '/content/drive/MyDrive/HIDE'
colab_target_path = 'demo_data/HIDE'

# 3. Create the symbolic link
if os.path.exists(drive_path):
    os.symlink(drive_path, colab_target_path)
    print("The HIDE dataset in Drive has been successfully linked to the project!")
else:
    print("ERROR: HIDE folder not found in Drive. Please check the path.")

In [ ]:
import os
import sys
import torch
import urllib.request

# 1. Add the project directories to the Python path
sys.path.append(os.getcwd())
sys.path.append(os.path.join(os.getcwd(), 'basicsr'))
from basicsr.models.archs.restormer_arch import Restormer

# 2. Download the pre-trained weights safely
os.makedirs('demo_data/pretrained_models', exist_ok=True)
weights_path = 'demo_data/pretrained_models/motion_deblurring.pth'

url = "https://github.com/swz30/Restormer/releases/download/v1.0/motion_deblurring.pth"
if not os.path.exists(weights_path):
    urllib.request.urlretrieve(url, weights_path)
    print("Weights file downloaded.")
else:
    print("Weights file already exists, skipping download.")

# 3. Initialize the model and move it to the GPU
model = Restormer()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 4. Load the weights into the model
checkpoint = torch.load(weights_path, map_location=device, weights_only=True)

if 'params' in checkpoint:
    model.load_state_dict(checkpoint['params'])
elif 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])
else:
    model.load_state_dict(checkpoint)

model.eval()

In [ ]:
import os
import random

# 1. Set the global random seed for reproducibility across all experiments
random.seed(42)

blur_dir = 'demo_data/HIDE/blur'
sharp_dir = 'demo_data/HIDE/sharp'

# 2. Retrieve all available images
all_images = [f for f in os.listdir(blur_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

# 3. Select 10 random images globally and save them to a fixed list
if len(all_images) < 10:
    print(f"ERROR: Not enough images found. Only {len(all_images)} available.")
    GLOBAL_SELECTED_IMAGES = all_images
else:
    GLOBAL_SELECTED_IMAGES = random.sample(all_images, 10)

for idx, img_name in enumerate(GLOBAL_SELECTED_IMAGES, 1):
    print(f"   {idx}. {img_name}")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import torch.nn.functional as F
import numpy as np
import random
import os

# Set a random seed for reproducibility
random.seed(42)

# 1. Retrieve the list of images and select 10 random files
blur_dir = 'demo_data/HIDE/blur'
image_list = [f for f in os.listdir(blur_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

if len(image_list) < 10:
    print(f"ERROR: Not enough images found. Found {len(image_list)}, but 10 are required.")
else:
    # Select 10 random images using the fixed seed
    selected_images = GLOBAL_SELECTED_IMAGES

    for image_name in selected_images:
        blur_path = os.path.join(blur_dir, image_name)
        sharp_path = os.path.join('demo_data/HIDE/sharp', image_name)

        # 2. Read images (OpenCV reads in BGR, we convert to RGB)
        img_blur = cv2.cvtColor(cv2.imread(blur_path), cv2.COLOR_BGR2RGB)
        img_sharp = cv2.cvtColor(cv2.imread(sharp_path), cv2.COLOR_BGR2RGB)

        # 3. Prepare for model input (Tensor and padding to multiples of 8)
        input_tensor = torch.from_numpy(img_blur).float().div(255.).permute(2, 0, 1).unsqueeze(0).to(device)

        h, w = input_tensor.shape[2], input_tensor.shape[3]
        factor = 8
        H, W = ((h + factor - 1) // factor) * factor, ((w + factor - 1) // factor) * factor
        padh = H - h if h % factor != 0 else 0
        padw = W - w if w % factor != 0 else 0
        input_tensor = F.pad(input_tensor, (0, padw, 0, padh), 'reflect')

        # 4. RUN THE MODEL
        with torch.no_grad():
            restored_tensor = model(input_tensor)

        # 5. Convert output back to image format (Numpy)
        restored_tensor = restored_tensor[:, :, :h, :w]
        restored_img = restored_tensor.permute(0, 2, 3, 1).cpu().detach().numpy().squeeze()
        restored_img = np.clip(restored_img, 0, 1)
        restored_img = (restored_img * 255).astype(np.uint8)

        # 6. Display the results in a 3-column format
        fig, axes = plt.subplots(1, 3, figsize=(20, 7))

        axes[0].imshow(img_blur)
        axes[0].set_title(f"1. Original Blurred Input\n({image_name})", fontsize=14, color='darkred', fontweight='bold')
        axes[0].axis('off')

        axes[1].imshow(restored_img)
        axes[1].set_title("2. Restormer Output\n(Restored Image)", fontsize=14, color='darkgreen', fontweight='bold')
        axes[1].axis('off')

        axes[2].imshow(img_sharp)
        axes[2].set_title("3. Ground Truth\n(Target Sharp Image)", fontsize=14, color='darkblue', fontweight='bold')
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

# 1. Create a dictionary to store the activations (attention signals)
attention_maps = {}

# 2. PyTorch Hook Function (copies the data as the model passes through that layer)
def get_attention(name):
    def hook(model, input, output):
        attention_maps[name] = output.detach()
    return hook

# 3. Attach the hook to the "Latent" (Bottleneck) block, the deepest layer of Restormer
hook_handle = model.latent[0].register_forward_hook(get_attention('latent_block_1'))

In [ ]:
import matplotlib.pyplot as plt
import cv2
import torch.nn.functional as F
import numpy as np
import random
import os
import torch

# 1. Retrieve the list of images and select 10 random files
if len(image_list) < 10:
    print(f"ERROR: Not enough images found. Found {len(image_list)}, but 10 are required.")
else:
    # Select 10 random images using the fixed seed
    selected_images = GLOBAL_SELECTED_IMAGES

    # 2. Setup the Hook for Attention Maps
    attention_maps = {}
    def get_attention(name):
        def hook(model, input, output):
            attention_maps[name] = output.detach()
        return hook

    # Attach the hook to the "Latent" (Bottleneck) block before starting the loop
    hook_handle = model.latent[0].register_forward_hook(get_attention('latent_block_1'))

    # 3. Process each image in the selected 10
    for image_name in selected_images:
        blur_path = os.path.join(blur_dir, image_name)

        # Read image and convert to RGB
        img_blur = cv2.cvtColor(cv2.imread(blur_path), cv2.COLOR_BGR2RGB)

        # Prepare tensor and apply padding
        input_tensor = torch.from_numpy(img_blur).float().div(255.).permute(2, 0, 1).unsqueeze(0).to(device)
        h, w = input_tensor.shape[2], input_tensor.shape[3]
        factor = 8
        H, W = ((h + factor - 1) // factor) * factor, ((w + factor - 1) // factor) * factor
        padh = H - h if h % factor != 0 else 0
        padw = W - w if w % factor != 0 else 0
        input_tensor = F.pad(input_tensor, (0, padw, 0, padh), 'reflect')

        # Run the model (The hook will capture the data in the background)
        with torch.no_grad():
            _ = model(input_tensor)

        # Retrieve the captured Attention data
        act = attention_maps['latent_block_1'] # Shape: [1, Channels, H, W]

        # Compute the spatial attention map by averaging across hundreds of channels
        heatmap = torch.mean(act, dim=1).squeeze().cpu().numpy()

        # Normalize the values between 0 and 1 (for visualization)
        heatmap = np.maximum(heatmap, 0)
        if np.max(heatmap) > 0:
            heatmap /= np.max(heatmap)

        # Resize the map to the original blurred image dimensions (without padding)
        heatmap_resized = cv2.resize(heatmap, (img_blur.shape[1], img_blur.shape[0]))

        # Apply the JET colormap (red/blue) to the heatmap
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

        # Superimpose the Heatmap onto the original image (60% Image, 40% Color)
        superimposed_img = cv2.addWeighted(img_blur, 0.6, heatmap_colored, 0.4, 0)

        # --- VISUALIZATION ---
        fig, axes = plt.subplots(1, 3, figsize=(20, 7))

        axes[0].imshow(img_blur)
        axes[0].set_title(f"1. Blurred Input\n({image_name})", fontsize=14, fontweight='bold')
        axes[0].axis('off')

        axes[1].imshow(heatmap_colored)
        axes[1].set_title("2. Pure Attention Map\n(Red: High Focus, Blue: Low Focus)", fontsize=14, fontweight='bold')
        axes[1].axis('off')

        axes[2].imshow(superimposed_img)
        axes[2].set_title("3. XAI Overlay on Image", fontsize=14, fontweight='bold')
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

    # 4. Remove the hook to free up memory and model resources after the loop is done
    hook_handle.remove()

In [ ]:
!pip install captum

In [ ]:
import torch
from captum.attr import IntegratedGradients
import matplotlib.pyplot as plt
import numpy as np
import cv2

# 1. Custom XAI Wrapper for Image Processing Models
class RestormerIGWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_tensor):
        output = self.model(input_tensor)

        # We analyze only a 40x40 pixel "target region" exactly at the center of the image.
        # This allows us to ask: "Where did you look in the blurred image to restore the object in the center?"
        h, w = output.shape[2], output.shape[3]
        center_h, center_w = h // 2, w // 2

        # We return the sum of the pixels in the target patch as a single scalar score
        target_patch = output[:, :, center_h-20:center_h+20, center_w-20:center_w+20]
        return target_patch.sum(dim=(1,2,3))

# Initialize the wrapper
wrapper_model = RestormerIGWrapper(model).to(device)
wrapper_model.eval()

# 2. Create the Baseline (Reference) Image
# Integrated Gradients steps gradually from zero (completely black) to our input image.
baseline = torch.zeros_like(input_tensor).to(device)

# 3. Enable gradient computation for our input tensor
input_tensor.requires_grad_()

In [ ]:
import gc
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# 1. Secure memory
torch.cuda.empty_cache()
gc.collect()

# 2. Custom XAI Wrapper for IG (Analyzes only a 4x4 target region in the center)
class RestormerIGMicroWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        h, w = out.shape[2], out.shape[3]
        center_h, center_w = h // 2, w // 2
        target_patch = out[:, :, center_h-2:center_h+2, center_w-2:center_w+2]
        return target_patch.sum(dim=(1,2,3))

wrapper_model = RestormerIGMicroWrapper(model).to(device)
wrapper_model.eval()

# Initialize Integrated Gradients
ig = IntegratedGradients(wrapper_model)

# 3. USE THE GLOBAL LIST DIRECTLY
for image_name in GLOBAL_SELECTED_IMAGES:

    # Read Image and cut a 256x256 "Micro-Patch"
    blur_path = os.path.join(blur_dir, image_name)
    img_blur = cv2.cvtColor(cv2.imread(blur_path), cv2.COLOR_BGR2RGB)

    patch_size = 256
    h, w, _ = img_blur.shape
    start_h = (h - patch_size) // 2
    start_w = (w - patch_size) // 2
    img_blur_patch = img_blur[start_h:start_h+patch_size, start_w:start_w+patch_size, :]

    input_patch_tensor = torch.from_numpy(img_blur_patch).float().div(255.).permute(2, 0, 1).unsqueeze(0).to(device)
    input_patch_tensor.requires_grad_()

    baseline = torch.zeros_like(input_patch_tensor).to(device)

    # Calculate IG
    attributions, delta = ig.attribute(input_patch_tensor,
                                       baseline,
                                       n_steps=5,
                                       internal_batch_size=1,
                                       return_convergence_delta=True)

    # TENSOR -> NUMPY STEP
    attr = attributions.squeeze().cpu().detach().numpy()
    attr = np.transpose(attr, (1, 2, 0))
    attr_gray = np.mean(np.abs(attr), axis=2)

    # Normalize and prevent division by zero
    max_val = np.max(attr_gray)
    if max_val > 0:
        attr_norm = attr_gray / max_val
    else:
        attr_norm = attr_gray

    # Visualization
    heatmap_ig = cv2.applyColorMap(np.uint8(255 * attr_norm), cv2.COLORMAP_MAGMA)
    heatmap_ig = cv2.cvtColor(heatmap_ig, cv2.COLOR_BGR2RGB)

    # Overlay
    superimposed_ig = cv2.addWeighted(img_blur_patch, 0.3, heatmap_ig, 0.7, 0)

    # Green target box in the center
    center_h, center_w = patch_size // 2, patch_size // 2
    cv2.rectangle(superimposed_ig, (center_w-2, center_h-2), (center_w+2, center_h+2), (0, 255, 0), 1)

    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_blur_patch)
    axes[0].set_title(f"Blurred Input ({patch_size}x{patch_size})", fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(heatmap_ig)
    axes[1].set_title("IG Attribution Map (Gradients)", fontweight='bold')
    axes[1].axis('off')

    axes[2].imshow(superimposed_ig)
    axes[2].set_title("Overlay (Green: Target)", fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    # Free memory to avoid Out of Memory (OOM) errors during the loop
    del input_patch_tensor, baseline, attributions
    torch.cuda.empty_cache()
    gc.collect()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from captum.attr import LayerGradCam
import os
import gc

# 1. Secure memory
torch.cuda.empty_cache()
gc.collect()

# 2. Define the Target Layer
# Grad-CAM requires gradients from a specific layer.
# We select the deepest (bottleneck) layer of Restormer, which is the latent block.
target_layer = model.latent[0]

# 3. Initialize Captum LayerGradCam
# We use the same 'wrapper_model' defined earlier (focused on the center).
layer_gc = LayerGradCam(wrapper_model, target_layer)

# Process each image in our global list
for image_name in GLOBAL_SELECTED_IMAGES:
    print(f"Processing: {image_name}")

    # --- Recreate image patch and tensor for the current image ---
    blur_path = os.path.join(blur_dir, image_name)
    img_blur = cv2.cvtColor(cv2.imread(blur_path), cv2.COLOR_BGR2RGB)

    patch_size = 256
    h, w, _ = img_blur.shape
    start_h = (h - patch_size) // 2
    start_w = (w - patch_size) // 2
    img_blur_patch = img_blur[start_h:start_h+patch_size, start_w:start_w+patch_size, :]

    input_patch_tensor = torch.from_numpy(img_blur_patch).float().div(255.).permute(2, 0, 1).unsqueeze(0).to(device)
    input_patch_tensor.requires_grad_()
    # -------------------------------------------------------------

    # 4. Calculate Attributions
    # Grad-CAM does not require a baseline or n_steps; it extracts gradients in a single pass.
    with torch.backends.cudnn.flags(enabled=False): # Disabling cudnn is sometimes necessary for LayerGradCam
        gc_attributions = layer_gc.attribute(input_patch_tensor)

    # 5. Process and Upsample the Output
    # The Grad-CAM output matches the size of the selected deep layer (lower resolution).
    # We upsample it to our original patch size (256x256) for visualization.
    upsampled_gc = F.interpolate(gc_attributions, size=(patch_size, patch_size), mode='bilinear', align_corners=False)
    attr_gc = upsampled_gc.squeeze().cpu().detach().numpy()

    # Keep only positive contributions (Grad-CAM inherently uses ReLU to show only relevant areas)
    attr_gc = np.maximum(attr_gc, 0)
    if np.max(attr_gc) > 0:
        attr_gc = attr_gc / np.max(attr_gc)

    # 6. Visualization (Grad-CAM is typically visualized with the JET colormap)
    heatmap_gc = cv2.applyColorMap(np.uint8(255 * attr_gc), cv2.COLORMAP_JET)
    heatmap_gc = cv2.cvtColor(heatmap_gc, cv2.COLOR_BGR2RGB)

    superimposed_gc = cv2.addWeighted(img_blur_patch, 0.4, heatmap_gc, 0.6, 0)

    # Green target box in the center
    center_h, center_w = patch_size // 2, patch_size // 2
    cv2.rectangle(superimposed_gc, (center_w-2, center_h-2), (center_w+2, center_h+2), (0, 255, 0), 1)

    # --- Plotting ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_blur_patch)
    axes[0].set_title(f"1. Blurred Input ({patch_size}x{patch_size})", fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(heatmap_gc)
    axes[1].set_title("2. Grad-CAM Map\n(Coarse and Low Resolution)", fontweight='bold')
    axes[1].axis('off')

    axes[2].imshow(superimposed_gc)
    axes[2].set_title("3. Overlay (Green: Target)", fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    # Free memory to prevent Out of Memory (OOM) errors during the loop
    del input_patch_tensor, gc_attributions, upsampled_gc
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as calculate_psnr
from skimage.metrics import structural_similarity as calculate_ssim
import numpy as np
import cv2
import os
import torch
import torch.nn.functional as F

# Initialize variables to hold the total scores
total_psnr_blur = 0
total_ssim_blur = 0
total_psnr_restored = 0
total_ssim_restored = 0
num_images = len(GLOBAL_SELECTED_IMAGES)

# Process each globally selected image
for image_name in GLOBAL_SELECTED_IMAGES:
    # Read the images
    blur_path = os.path.join(blur_dir, image_name)
    sharp_path = os.path.join(sharp_dir, image_name)

    img_blur = cv2.cvtColor(cv2.imread(blur_path), cv2.COLOR_BGR2RGB)
    img_sharp = cv2.cvtColor(cv2.imread(sharp_path), cv2.COLOR_BGR2RGB)

    # --- Run Inference to get the Restored Image ---
    input_tensor = torch.from_numpy(img_blur).float().div(255.).permute(2, 0, 1).unsqueeze(0).to(device)
    h, w = input_tensor.shape[2], input_tensor.shape[3]
    factor = 8
    H, W = ((h + factor - 1) // factor) * factor, ((w + factor - 1) // factor) * factor
    padh = H - h if h % factor != 0 else 0
    padw = W - w if w % factor != 0 else 0
    input_tensor = F.pad(input_tensor, (0, padw, 0, padh), 'reflect')

    with torch.no_grad():
        restored_tensor = model(input_tensor)

    restored_tensor = restored_tensor[:, :, :h, :w]
    restored_img = restored_tensor.permute(0, 2, 3, 1).cpu().detach().numpy().squeeze()
    restored_img = np.clip(restored_img, 0, 1)
    restored_img = (restored_img * 255).astype(np.uint8)
    # -----------------------------------------------

    # 1. Ensure dimensions match exactly (to avoid padding/cropping metric errors)
    height, width, _ = img_sharp.shape
    img_blur_resized = cv2.resize(img_blur, (width, height))
    restored_img_resized = cv2.resize(restored_img, (width, height))

    # 2. CALCULATE METRICS FOR THE BLURRED IMAGE (Baseline)
    psnr_blur = calculate_psnr(img_sharp, img_blur_resized, data_range=255)
    ssim_blur = calculate_ssim(img_sharp, img_blur_resized, data_range=255, channel_axis=2)

    total_psnr_blur += psnr_blur
    total_ssim_blur += ssim_blur

    # 3. CALCULATE METRICS FOR THE RESTORMER OUTPUT (Our Success)
    psnr_restored = calculate_psnr(img_sharp, restored_img_resized, data_range=255)
    ssim_restored = calculate_ssim(img_sharp, restored_img_resized, data_range=255, channel_axis=2)

    total_psnr_restored += psnr_restored
    total_ssim_restored += ssim_restored

# Calculate Averages
avg_psnr_blur = total_psnr_blur / num_images
avg_ssim_blur = total_ssim_blur / num_images
avg_psnr_restored = total_psnr_restored / num_images
avg_ssim_restored = total_ssim_restored / num_images

# 4. CALCULATE OVERALL IMPROVEMENT
psnr_improvement = avg_psnr_restored - avg_psnr_blur
ssim_improvement = avg_ssim_restored - avg_ssim_blur

# Print the final benchmark results
print(f"Average Scores for Blurred (Original) Images:")
print(f"   - PSNR: {avg_psnr_blur:.2f} dB")
print(f"   - SSIM: {avg_ssim_blur:.4f}\n")

print(f"Average Scores for Restormer (Restored) Images:")
print(f"   - PSNR: {avg_psnr_restored:.2f} dB")
print(f"   - SSIM: {avg_ssim_restored:.4f}\n")

print(f"MATHEMATICAL IMPROVEMENT PROVIDED BY THE MODEL:")
print(f"   - PSNR Increase: +{psnr_improvement:.2f} dB")
print(f"   - SSIM Increase: +{ssim_improvement:.4f}")

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as calculate_psnr
from skimage.metrics import structural_similarity as calculate_ssim
from captum.attr import LayerGradCam, IntegratedGradients
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import os
import gc

# Initialize variables to accumulate the total quality drops
deletion_ratio = 10
total_ig_drop = 0
total_gc_drop = 0
num_images = len(GLOBAL_SELECTED_IMAGES)

# Prepare models/hooks globally before the loop
target_layer = model.latent[0]
layer_gc = LayerGradCam(wrapper_model, target_layer)
ig = IntegratedGradients(wrapper_model)

for image_name in GLOBAL_SELECTED_IMAGES:

    # 1. READ IMAGE AND PREPARE PATCH
    blur_path = os.path.join(blur_dir, image_name)
    sharp_path = os.path.join(sharp_dir, image_name)

    img_blur = cv2.cvtColor(cv2.imread(blur_path), cv2.COLOR_BGR2RGB)
    img_sharp = cv2.cvtColor(cv2.imread(sharp_path), cv2.COLOR_BGR2RGB)

    patch_size = 256
    h, w, _ = img_blur.shape
    start_h, start_w = (h - patch_size) // 2, (w - patch_size) // 2

    img_blur_patch = img_blur[start_h:start_h+patch_size, start_w:start_w+patch_size, :]
    img_sharp_patch_np = img_sharp[start_h:start_h+patch_size, start_w:start_w+patch_size, :]

    input_patch_tensor = torch.from_numpy(img_blur_patch).float().div(255.).permute(2, 0, 1).unsqueeze(0).to(device)
    input_patch_tensor.requires_grad_()

    # 2. GROUND TRUTH AND BASELINE PREPARATION
    # First, we measure how well the model performs normally without masking (Baseline).
    with torch.no_grad():
        restored_baseline_tensor = model(input_patch_tensor)
        restored_baseline = restored_baseline_tensor.squeeze().permute(1, 2, 0).cpu().numpy()
        restored_baseline = (np.clip(restored_baseline, 0, 1) * 255.0).astype(np.uint8)

    baseline_psnr = calculate_psnr(img_sharp_patch_np, restored_baseline, data_range=255)

    # 3. GENERATE IG MAP FOR CURRENT IMAGE
    baseline_tensor = torch.zeros_like(input_patch_tensor).to(device)
    attributions, _ = ig.attribute(input_patch_tensor, baseline_tensor, n_steps=5, internal_batch_size=1, return_convergence_delta=True)
    attr = attributions.squeeze().cpu().detach().numpy()
    attr = np.transpose(attr, (1, 2, 0))
    attr_gray = np.mean(np.abs(attr), axis=2)
    max_val = np.max(attr_gray)
    attr_norm = attr_gray / max_val if max_val > 0 else attr_gray

    # 4. GENERATE GRAD-CAM MAP FOR CURRENT IMAGE
    # We disable cudnn to prevent potential compatibility issues with LayerGradCam
    with torch.backends.cudnn.flags(enabled=False):
        gc_attributions = layer_gc.attribute(input_patch_tensor)
    upsampled_gc = F.interpolate(gc_attributions, size=(patch_size, patch_size), mode='bilinear', align_corners=False)
    attr_gc = upsampled_gc.squeeze().cpu().detach().numpy()
    attr_gc = np.maximum(attr_gc, 0)
    if np.max(attr_gc) > 0:
        attr_gc = attr_gc / np.max(attr_gc)

    # 5. DELETION MASKING FUNCTION
    def apply_deletion_mask(input_tensor, heatmap, k_percent):
        threshold = np.percentile(heatmap, 100 - k_percent)
        mask = heatmap >= threshold
        mask_tensor = torch.from_numpy(mask).to(input_tensor.device).unsqueeze(0).unsqueeze(0)

        masked_input = input_tensor.clone()
        masked_input.masked_fill_(mask_tensor, 0.0) # Black out the pixels
        return masked_input, mask

    # 6. DELETION TEST FOR IG
    masked_input_ig, mask_ig = apply_deletion_mask(input_patch_tensor, attr_norm, k_percent=deletion_ratio)
    with torch.no_grad():
        restored_ig_masked = model(masked_input_ig).squeeze().permute(1, 2, 0).cpu().numpy()
        restored_ig_masked = (np.clip(restored_ig_masked, 0, 1) * 255.0).astype(np.uint8)

    psnr_ig_drop = calculate_psnr(img_sharp_patch_np, restored_ig_masked, data_range=255)
    ig_psnr_loss = baseline_psnr - psnr_ig_drop
    total_ig_drop += ig_psnr_loss

    # 7. DELETION TEST FOR GRAD-CAM
    masked_input_gc, mask_gc = apply_deletion_mask(input_patch_tensor, attr_gc, k_percent=deletion_ratio)
    with torch.no_grad():
        restored_gc_masked = model(masked_input_gc).squeeze().permute(1, 2, 0).cpu().numpy()
        restored_gc_masked = (np.clip(restored_gc_masked, 0, 1) * 255.0).astype(np.uint8)

    psnr_gc_drop = calculate_psnr(img_sharp_patch_np, restored_gc_masked, data_range=255)
    gc_psnr_loss = baseline_psnr - psnr_gc_drop
    total_gc_drop += gc_psnr_loss

    # 8. VISUALIZATION
    # Draw the masked (corrupted) inputs
    img_blur_patch_np = img_blur_patch.astype(np.float32) / 255.0
    mask_ig_vis = np.expand_dims(mask_ig, axis=-1)
    mask_gc_vis = np.expand_dims(mask_gc, axis=-1)

    # Make the masked areas completely black
    vis_masked_ig = (img_blur_patch_np * (1 - mask_ig_vis) * 255).astype(np.uint8)
    vis_masked_gc = (img_blur_patch_np * (1 - mask_gc_vis) * 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    axes[0].imshow(vis_masked_ig)
    axes[0].set_title(f"Pixels Deleted by IG ({deletion_ratio}%)\nQuality Loss: -{ig_psnr_loss:.2f} dB", fontweight='bold', color='darkred')
    axes[0].axis('off')

    axes[1].imshow(vis_masked_gc)
    axes[1].set_title(f"Pixels Deleted by Grad-CAM ({deletion_ratio}%)\nQuality Loss: -{gc_psnr_loss:.2f} dB", fontweight='bold', color='darkorange')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    # Free memory to prevent Out of Memory (OOM) errors
    del input_patch_tensor, baseline_tensor, attributions, gc_attributions, masked_input_ig, masked_input_gc
    torch.cuda.empty_cache()
    gc.collect()

# 9. FINAL EVALUATION AVERAGES
avg_ig_drop = total_ig_drop / num_images
avg_gc_drop = total_gc_drop / num_images

print("\nFINAL EVALUATION (10-IMAGE AVERAGE):")
print(f"   - Average PSNR Drop from IG Deletion: -{avg_ig_drop:.2f} dB")
print(f"   - Average PSNR Drop from Grad-CAM Deletion: -{avg_gc_drop:.2f} dB\n")

if avg_ig_drop > avg_gc_drop:
    print("Integrated Gradients (IG) is MORE FAITHFUL to the model's decision mechanism.")
    print("By deleting the correct pixels, it successfully degraded the model's performance more than Grad-CAM.")
else:
    print("Grad-CAM caused a higher performance drop on average.")